
# Reprodutibilidade e diagnóstico das dependências do PySITS no Windows
## 1. Objetivo

Este notebook tem como objetivo reproduzir o processo de instalação e execução do PySITS em ambiente Windows, utilizando um ambiente Conda limpo e isolado.

Busca-se identificar possíveis problemas de compatibilidade entre as dependências, registrar os erros encontrados durante a instalação e execução do pacote e documentar as modificações necessárias para viabilizar seu funcionamento.

## 2. Ambiente experimental

O experimento é conduzido em sistema operacional Windows, utilizando o gerenciador de ambientes Miniconda e o VS Code como interface de desenvolvimento.

```bash
conda env create pysits_windows

conda activate pysits_windows
```

## 3. Metodologia

A investigação segue os passos:

1. Caracterização do ambiente computacional disponível.
2. Reprodução do procedimento original de instalação do PySITS.
3. Identificação e registro de erros e incompatibilidades.
4. Investigação das possíveis causas dos problemas encontrados.

## 4. Configuração original do ambiente

Inicialmente, foram registradas as versões do sistema operacional, do Python, do gerenciador de ambientes Conda e do R, além das principais informações sobre a configuração do ambiente experimental.

Para isso, foram executados os seguintes comandos no terminal:

```bash
python --version

python -c "import sys, platform; print('Executable:', sys.executable); print('Architecture:', platform.architecture()); print('Platform:', platform.platform())"

conda --version

conda list

python -m pip --version
```

Para verificar a instalação e o funcionamento básico do R, foi executado:

```powershell
"C:\Program Files\R\R-4.6.2\bin\x64\R.exe" --vanilla -e "cat('R version:', R.version.string, '\n'); cat('R_HOME:', R.home(), '\n'); library(stats); cat('STATS_OK\n')"
```

**Configuração inicial identificada:**

- **Sistema operacional:** Windows 11.
- **Python:** 3.12.16.
- **Arquitetura:** 64-bit.
- **Gerenciador de ambientes:** Miniconda 26.9.4.
- **pip:** 26.2.2.
- **Ambiente Conda:** `pysits_windows`.
- **Dependências Python:** ambiente inicialmente com poucas dependências instaladas.
- **R:** 4.6.2 (2026-10-29 ucrt) — "Diversified Portfolio".
- **R_HOME:** `C:/PROGRA~1/R/R-46~1.2`.
- **Teste da biblioteca stats:** executado com sucesso (`STATS_OK`).

## 5. Reprodução da instalação original do PySITS

O procedimento de instalação do PySITS foi reproduzido seguindo as etapas abaixo:

**1. Inicialização do R**

O console R foi iniciado a partir do terminal Conda, utilizando o caminho completo do executável:

```cmd
"C:\Program Files\R\R-4.6.2\bin\x64\R.exe"
```

**2. Instalação das dependências no R**

No console R, foram executados os seguintes comandos:

```r
install.packages("sits")
install.packages("arrow")
```

**3. Verificação das instalações**

Para verificar se os pacotes foram instalados e carregados corretamente, foram executados:

```r
library(sits)
library(arrow)

packageVersion("sits")
packageVersion("arrow")
```

As versões instaladas foram:

- `sits`: 1.5.4
- `arrow`: 26.0.1

**4. Encerramento do R**

Após a verificação, o console R foi encerrado:

```r
q()
```

**5. Instalação do PySITS**

De volta ao terminal Conda, com o ambiente `pysits_windows` ativo, foi executado:

```bash
pip install pysits
```

**Observação sobre o ambiente Jupyter**

Para permitir a utilização do ambiente `pysits_windows` como kernel no Jupyter Notebook, foi executado:

```bash
conda install -n pysits_windows ipykernel --update-deps --force-reinstall
```

Essa etapa está relacionada à configuração do ambiente de desenvolvimento, não sendo um requisito específico da instalação do PySITS.

## 6. Importação do PySITS
Após a instalação do pacote, foi realizada a primeira tentativa de importação do PySITS no Jupyter Notebook:

In [1]:
from pysits import *
print("PySITS OK")

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 233: invalid continuation byte <traceback object at 0x000001E91FDD6980>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 180: invalid continuation byte <traceback object at 0x000001E91A330980>


RRuntimeError: Error in inDL(x, as.logical(local), as.logical(now), ...) : 
  unable to load shared object 'C:/Program Files/R/R-4.6.2/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  Não foi possível encontrar o módulo especificado.


**Resultado: falha na importação.**

A execução retornou um erro `RRuntimeError`, relacionado ao carregamento da biblioteca `stats.dll`, pertencente à instalação do R 4.6.2.

Trecho principal da mensagem de erro:

```text
RRuntimeError: Error in inDL(x, as.logical(local), as.logical(now), ...) :
unable to load shared object
'C:/Program Files/R/R-4.6.2/library/stats/libs/x64/stats.dll':

LoadLibrary failure:
Não foi possível encontrar o módulo especificado.
```

**Observação:** Embora o pacote `stats` tenha sido carregado com sucesso durante o teste inicial diretamente no R, sua inicialização apresentou falha durante a importação do PySITS, por meio da integração com Python utilizando `rpy2`.

O erro indica um problema no carregamento da biblioteca compartilhada do R durante a execução integrada. A causa ainda precisa ser investigada.

## 7. Diagnóstico dos erros encontrados

### 7.1. Investigar os pacotes R

1. Verificar se o rpy2 consegue inicializar o R de forma independente:

In [2]:
import rpy2.robjects as ro

print(ro.r("R.version.string"))

[1] "R version 4.6.2 (2026-10-29 ucrt)"



**Resultado:** o R 4.6.2 foi inicializado corretamente por meio do rpy2.

2. Verificar se o rpy2 consegue carregar `stats`

In [3]:
from rpy2.robjects.packages import importr

stats = importr("stats")

print("stats loaded successfully")

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 234: invalid continuation byte <traceback object at 0x000001E9210A7680>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 180: invalid continuation byte <traceback object at 0x000001E9210A7800>


RRuntimeError: Error in inDL(x, as.logical(local), as.logical(now), ...) : 
  unable to load shared object 'C:/Program Files/R/R-4.6.2/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  Não foi possível encontrar o módulo especificado.


**Resultado:** A execução retornou um RRuntimeError, indicando que o Windows não conseguiu carregar a biblioteca stats.dll, pertencente à instalação do R 4.6.2.

3. Verificar se o rpy2 consegue carregar `sits`

In [4]:
sits = importr("sits")

print("sits loaded successfully")

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 233: invalid continuation byte <traceback object at 0x000001E9214D8540>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 180: invalid continuation byte <traceback object at 0x000001E9214D8740>


RRuntimeError: Error in inDL(x, as.logical(local), as.logical(now), ...) : 
  unable to load shared object 'C:/Program Files/R/R-4.6.2/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  Não foi possível encontrar o módulo especificado.


**Resultado:** a execução retornou um RRuntimeError, relacionado ao carregamento de stats.dll. Foram observadas adicionalmente mensagens UnicodeDecodeError durante a comunicação entre R e Python.

**Síntese dos resultados**

A inicialização do R pelo rpy2 ocorreu normalmente, porém o carregamento dos pacotes stats e sits apresentou falhas. Os resultados indicam um problema durante o carregamento de bibliotecas compartilhadas do R na execução integrada com Python, cuja causa ainda precisa ser investigada.

**4. Verificação da biblioteca compartilhada e das variáveis de ambiente**

Inicialmente, verificou-se a existência do arquivo `stats.dll` no diretório da instalação do R.

**Resultado:** o arquivo foi localizado em:

`C:\Program Files\R\R-4.6.2\library\stats\libs\x64\stats.dll`

Em seguida, foram verificadas as variáveis de ambiente do sistema:

- `where.exe R.exe`: não localizou o executável do R.
- `R_HOME`: não estava definido no ambiente Conda.
- `PATH`: não incluía os diretórios de execução da instalação do R.

Os resultados indicam que o executável do R não estava disponível diretamente pelo PATH do ambiente Conda, embora pudesse ser iniciado utilizando seu caminho absoluto.

**Próxima etapa:** testar a inclusão temporária dos diretórios do R nas variáveis `PATH` e `R_HOME`, verificando se a configuração permite o carregamento dos pacotes por meio do `rpy2`.

**5. Teste com configuração temporária das variáveis de ambiente**

Considerando que `R_HOME` não estava definido e que os diretórios do R não estavam presentes no `PATH` do ambiente Conda, foram adicionados temporariamente os caminhos da instalação do R às variáveis de ambiente.

Após essa configuração, foram repetidos os testes de inicialização do R e carregamento do pacote `stats` por meio do `rpy2`.

**Resultados:**

- Inicialização do R 4.6.2: sucesso.
- Carregamento do pacote `stats`: sucesso.
- Mensagem adicional: `'sh' não é reconhecido como um comando interno ou externo`.

A configuração temporária permitiu o carregamento de `stats`, indicando que as variáveis de ambiente estavam relacionadas à falha inicial. Todavia, a importação completa do PySITS continuou apresentando um erro relacionado à compatibilidade com o pandas.

**6. Resultados após a configuração das variáveis de ambiente**

Após a configuração temporária de `R_HOME` e `PATH`, os testes de integração entre Python e R foram repetidos.

In [5]:
import os
import sys
import subprocess

env = os.environ.copy()

R_HOME = r"C:\Program Files\R\R-4.6.2"

env["R_HOME"] = R_HOME
env["PATH"] = (
    R_HOME + r"\bin\x64;" +
    R_HOME + r"\bin;" +
    env["PATH"]
)

print("Python executable:", sys.executable)
print("R_HOME:", env["R_HOME"])

Python executable: c:\Users\sofia\miniconda3\envs\pysits_windows\python.exe
R_HOME: C:\Program Files\R\R-4.6.2


In [6]:
def run_test(description, code):
    print(f"\n{'='*60}")
    print(description)
    print('='*60)

    result = subprocess.run(
        [sys.executable, "-c", code],
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print("\nSTDOUT:")
    print(result.stdout or "(empty)")

    print("\nSTDERR:")
    print(result.stderr or "(empty)")

    print("\nReturn code:", result.returncode)

    return result

In [7]:
test_r = run_test(
    "Test 1 - R initialization through rpy2",
    """
import rpy2.robjects as ro
print(ro.r("R.version.string"))
"""
)


Test 1 - R initialization through rpy2

STDOUT:
[1] "R version 4.6.2 (2026-10-29 ucrt)"





STDERR:
'sh' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


Return code: 0


In [8]:
test_stats = run_test(
    "Test 2 - Loading stats through rpy2",
    """
from rpy2.robjects.packages import importr

stats = importr("stats")
print("stats loaded successfully")
"""
)


Test 2 - Loading stats through rpy2

STDOUT:
stats loaded successfully



STDERR:
'sh' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


Return code: 0


In [9]:
test_sits = run_test(
    "Test 3 - Loading sits through rpy2",
    """
from rpy2.robjects.packages import importr

sits = importr("sits")
print("sits loaded successfully")
"""
)


Test 3 - Loading sits through rpy2

STDOUT:
sits loaded successfully



STDERR:
'sh' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


Return code: 0


In [10]:
test_pysits = run_test(
    "Test 4 - Importing PySITS",
    """
import pysits
print("PySITS imported successfully")
"""
)


Test 4 - Importing PySITS

STDOUT:
(empty)

STDERR:
'sh' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.
Traceback (most recent call last):
  File "<string>", line 2, in <module>
  File "c:\Users\sofia\miniconda3\envs\pysits_windows\Lib\site-packages\pysits\__init__.py", line 23, in <module>
    from .sits.classification import sits_classify, sits_label_classification, sits_smooth
  File "c:\Users\sofia\miniconda3\envs\pysits_windows\Lib\site-packages\pysits\sits\classification.py", line 21, in <module>
    from pysits.conversions.decorators import function_call
  File "c:\Users\sofia\miniconda3\envs\pysits_windows\Lib\site-packages\pysits\conversions\decorators.py", line 25, in <module>
    from pysits.conversions.common import convert_to_r, fix_reserved_words_parameters
  File "c:\Users\sofia\miniconda3\envs\pysits_windows\Lib\site-packages\pysits\conversions\common.py", line 32, in <module>
    from pysits.conversions.tibble import

**Resultados:**

| Teste | Resultado |
|---|---|
| Inicialização do R pelo `rpy2` | Sucesso |
| Carregamento de `stats` | Sucesso |
| Carregamento de `sits` | Sucesso |
| Importação do PySITS | Falha |

Embora os três primeiros testes tenham sido executados com sucesso, a importação do PySITS apresentou um novo erro:

```text
ImportError: cannot import name 'Self' from 'pandas._typing'
```

A falha ocorreu durante a importação do módulo `pysits.models.frame`, indicando um possível problema de compatibilidade entre a implementação do PySITS e a versão instalada do pandas.

Dessa forma, foram identificados dois problemas distintos durante a reprodução:

1. Configuração das variáveis de ambiente do R, que impedia o carregamento de bibliotecas compartilhadas durante a integração com Python.
2. Compatibilidade entre o PySITS e a versão instalada do pandas, identificada após a resolução do primeiro problema.

### 7.2. Investigar as dependências do Python
**1. Investigar o pandas**

Foi verificada as versões do Python e pandas instalado no ambiente, além da localização do pacote.

Foram consultadas as dependências declaradas pelo PySITS 1.5.4 em seus metadados. A versão mínima exigida para o pandas é 2.2.3, sem especificação de uma versão máxima.

Foi verificada a disponibilidade do objeto Self no módulo interno pandas._typing, utilizado durante a importação do PySITS.

In [11]:
import pandas as pd
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Pandas location:", pd.__file__)

Python: 3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]
Pandas: 3.1.1
Pandas location: c:\Users\sofia\miniconda3\envs\pysits_windows\Lib\site-packages\pandas\__init__.py


In [14]:
# Verificar as dependências do PySITS
from importlib.metadata import metadata, requires

print("PySITS version:", metadata("pysits")["Version"])
print("\nDeclared dependencies:")

for dependency in requires("pysits") or []:
    print(dependency)

PySITS version: 1.5.4

Declared dependencies:
geopandas>=1.1.0
jinja2>=3.1.6
matplotlib>=3.10.1
pandas>=2.2.3
pillow>=11.1.0
pyarrow>=20.0.0
rpy2<4.0.0,>=3.5.17
shapely>=2.1.1
cloudpickle>=3.1.1; extra == 'dev'
pre-commit>=3.6.0; extra == 'dev'
pytest-cov>=6.1.1; extra == 'dev'
pytest-randomly>=3.16.0; extra == 'dev'
pytest>=7.4.4; extra == 'dev'
ruff>=0.11.12; extra == 'dev'
affine>=2.4.0; extra == 'xarray'
dask>=2025.3.0; extra == 'xarray'
rioxarray>=0.18.2; extra == 'xarray'
xarray>=2025.3.0; extra == 'xarray'


In [13]:
import pandas._typing as pandas_typing

print("Self available:", hasattr(pandas_typing, "Self"))

Self available: False


**Resultados:**

- Python: 3.12.16.
- pandas: 3.1.1.
- PySITS: 1.5.4.
- Dependência declarada: `pandas>=2.2.3`.
- Disponibilidade de `Self` em `pandas._typing`: não.

A ausência do objeto `Self` no módulo `pandas._typing` está relacionada ao erro observado durante a importação do PySITS. Embora a versão instalada do pandas satisfaça a dependência mínima declarada, há uma incompatibilidade entre a implementação do PySITS e a versão instalada do pandas.

**2. Teste de uma versão diferente do pandas**

Foi realizado um teste com a versão mínima do pandas indicada pelo PySITS 1.5.4.

```bash
python -m pip install pandas==2.2.3
```

O kernel do Jupyter foi reiniciado e foram repetidos os testes de verificação da versão do pandas e da disponibilidade do objeto Self no módulo pandas._typing.

In [1]:
import pandas as pd
print("Pandas:", pd.__version__)

import pandas._typing as pandas_typing
print("Self available:", hasattr(pandas_typing, "Self"))

Pandas: 2.2.3
Self available: True


**Resultados:**

- Versão do pandas: 2.2.3.
- Disponibilidade de `Self` em `pandas._typing`: sim.

O objeto `Self` está disponível no pandas 2.2.3, diferentemente do observado na versão 3.1.1. Essa alteração resolve a incompatibilidade identificada nessa etapa da investigação.

**3. Importar o PySITS novamente**

Após a alteração da versão do pandas para 2.2.3, foi repetido o teste de importação do PySITS.

In [2]:
from pysits import *
print("PySITS OK")

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 233: invalid continuation byte <traceback object at 0x000001AE644FD080>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xe3 in position 180: invalid continuation byte <traceback object at 0x000001AE64475D40>


RRuntimeError: Error in inDL(x, as.logical(local), as.logical(now), ...) : 
  unable to load shared object 'C:/Program Files/R/R-4.6.2/library/stats/libs/x64/stats.dll':
  LoadLibrary failure:  Não foi possível encontrar o módulo especificado.


**Resultado:** A importação continuou apresentando erro. O erro relacionado à ausência do objeto `Self` não foi mais observado. Entretanto, ocorreu uma falha durante o carregamento do pacote R `sits` por meio do `rpy2`, com a seguinte mensagem:

`unable to load shared object ... stats.dll: LoadLibrary failure`

Também foram observadas mensagens adicionais de `UnicodeDecodeError` durante os callbacks do R.

O resultado indica que a alteração da versão do pandas resolveu o problema de compatibilidade identificado anteriormente, mas não foi suficiente para permitir a importação completa do PySITS.

### 7.3. Integração Python-R
**1. Verificar os caminhos do Python e do R**

Após a falha na importação completa do PySITS, foram verificadas as variáveis de ambiente utilizadas pelo kernel do Jupyter.

In [3]:
import os
import sys

print("Python:", sys.executable)
print("R_HOME:", os.environ.get("R_HOME"))
print("R in PATH:", any(
    "R-4.6.2" in path for path in os.environ.get("PATH", "").split(";")
))

Python: c:\Users\sofia\miniconda3\envs\pysits_windows\python.exe
R_HOME: C:/PROGRA~1/R/R-46~1.2
R in PATH: False


**Resultados:**

- Executável Python: localizado no ambiente Conda `pysits_windows`.
- `R_HOME`: definido como `C:/PROGRA~1/R/R-46~1.2`.
- Diretórios do R no `PATH`: não identificados.

Embora a variável `R_HOME` esteja definida, os diretórios da instalação do R não estão presentes no `PATH` do kernel. Essa diferença pode estar relacionada à falha no carregamento das bibliotecas compartilhadas do R durante a importação do PySITS.

**2. Configurar temporariamente as variáveis de ambiente**

Como os diretórios do R não estavam presentes no `PATH` do kernel do Jupyter, foi realizada uma configuração temporária das variáveis de ambiente diretamente no notebook.

In [4]:
import os

R_HOME = r"C:\Program Files\R\R-4.6.2"

os.environ["R_HOME"] = R_HOME
os.environ["PATH"] = (
    R_HOME + r"\bin\x64;" +
    R_HOME + r"\bin;" +
    os.environ["PATH"]
)

print("R_HOME:", os.environ["R_HOME"])
print("R in PATH:", any(
    "R-4.6.2" in path
    for path in os.environ["PATH"].split(";")
))

R_HOME: C:\Program Files\R\R-4.6.2
R in PATH: True


**Resultados:**

- `R_HOME`: definido como `C:\Program Files\R\R-4.6.2`.
- Diretórios do R no `PATH`: identificados após a configuração.

**3. Repetir a importação do PySITS**

In [5]:
from pysits import *
print("PySITS OK")

PySITS OK


**Resultado:** 
- Importação do PySITS: sucesso.

Após a configuração temporária das variáveis de ambiente, a importação completa do PySITS foi realizada com sucesso, utilizando Python 3.12.16, R 4.6.2 e pandas 2.2.3.

### 7.4. Teste funcional do PySITS

Após a importação bem-sucedida do PySITS, foi executado um exemplo prático previamente utilizado no ambiente JupyterLab, com o objetivo de verificar o funcionamento de suas principais operações.

O teste consistiu na criação de um cubo de dados Sentinel-2 NDVI, utilizando a coleção `SENTINEL-2-16D` do Brazil Data Cube (BDC), para o período de 01/01/2023 a 31/12/2023 e uma pequena região de interesse (ROI).

Em seguida, foram executadas as funções `sits_cube()` e `sits_timeline()` para criar o cubo e consultar sua cobertura temporal.

In [7]:
from pysits import sits_cube, sits_timeline
# Define the analysis period
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"

# Define a small region of interest (ROI)
ROI = {
    "xmin": -46.0,
    "xmax": -45.9,
    "ymin": -23.0,
    "ymax": -22.9
}

# Create a Sentinel-2 NDVI data cube
cube = sits_cube(
    source="BDC", # necessário senha para usuários externos, checar: https://brazildatacube.dpi.inpe.br/ 
    collection="SENTINEL-2-16D",
    bands=["NDVI"],
    start_date=START_DATE,
    end_date=END_DATE,
    roi=ROI,
    multicores=1
)

# Inspect the data cube
print(cube)

# Inspect the temporal coverage
print(sits_timeline(cube))


  |======================================================================| 100%
  source      collection   satellite sensor    tile       xmin       xmax  \
0    BDC  SENTINEL-2-16D  SENTINEL-2    MSI  030030  5792000.0  5897600.0   

        ymin       ymax                                                crs  \
0  8680000.0  8785600.0  PROJCRS["unknown",\n    BASEGEOGCRS["unknown",...   

                                           file_info  
0                            fid        date  ban...  
['2023-01-01', '2023-01-17', '2023-02-02', '2023-02-18', '2023-03-06', '2023-03-22', '2023-04-07', '2023-04-23', '2023-05-09', '2023-05-25', '2023-06-10', '2023-06-26', '2023-07-12', '2023-07-28', '2023-08-13', '2023-08-29', '2023-09-14', '2023-09-30', '2023-10-16', '2023-11-01', '2023-11-17', '2023-12-03', '2023-12-19']


**Resultados:**

- Criação do cubo de dados: sucesso.
- Fonte: BDC.
- Coleção: `SENTINEL-2-16D`.
- Satélite: Sentinel-2.
- Sensor: MSI.
- Tile: 030030.
- Cobertura temporal: 23 datas, entre 01/01/2023 e 19/12/2023.

O teste funcional foi executado com sucesso, demonstrando que o PySITS foi importado corretamente e conseguiu executar operações de acesso e consulta aos dados do BDC no ambiente Windows.

# 8. Considerações finais

Os testes realizados no ambiente Windows permitiram identificar e solucionar problemas relacionados às dependências Python e à integração entre Python e R durante a configuração e utilização do PySITS 1.5.4.

A importação do PySITS apresentou uma incompatibilidade relacionada ao pandas 3.1.1, em que o objeto `Self`, importado pelo pacote a partir do módulo `pandas._typing`, não estava disponível nessa versão. Após a instalação do pandas 2.2.3, o objeto passou a estar disponível e o erro deixou de ocorrer.

Foi identificada uma falha na integração entre Python e R, relacionada à ausência dos diretórios da instalação do R na variável de ambiente `PATH` do kernel do Jupyter. A configuração temporária de `R_HOME` e `PATH` permitiu o carregamento dos pacotes R necessários e a importação completa do PySITS.

Por fim, foi realizado um teste funcional utilizando as funções `sits_cube()` e `sits_timeline()`, com dados da coleção `SENTINEL-2-16D` do Brazil Data Cube (BDC). A execução foi bem-sucedida, demonstrando que o ambiente configurado foi capaz de acessar os dados, realizar operações de criação e consulta de cubos de dados.

Os testes demonstraram a viabilidade da utilização do PySITS 1.5.4 no ambiente Windows com:
- Python 3.12.16
- R 4.6.2 
- pandas 2.2.3
Todavia, as variáveis de ambiente necessárias à integração Python–R devem ser configuradas adequadamente.

Recomenda-se investigar uma configuração permanente dessas variáveis de ambiente e ampliar os testes funcionais, de modo a avaliar outras operações disponibilizadas pelo PySITS.